# Chapter 14: Advanced Panel Data Methods

Topics: fixed effects, random effects, Hausman test, clustered standard errors.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import wooldridge as woo
import matplotlib.pyplot as plt
import os
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS, compare

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## Data Setup

## Step 2 – Panel Data Model

**General unobserved effects model:**
$$y_{it} = \mathbf{x}_{it}'\boldsymbol{\beta} + a_i + u_{it}, \quad i=1,\ldots,N; \; t=1,\ldots,T$$

where $a_i$ is the **unobserved individual effect** (captures all time-constant unobservables for individual $i$).

**Key question:** Is $a_i$ correlated with $\mathbf{x}_{it}$?
- If **no** ($\text{Cov}(a_i, \mathbf{x}_{it})=0$): Random Effects (RE) is consistent and efficient
- If **yes** ($\text{Cov}(a_i, \mathbf{x}_{it})\neq 0$): Fixed Effects (FE) is required

In [ ]:
df = woo.dataWoo('wagepan')
df = df.set_index(['nr', 'year'])
print(f"Panel dimensions: {df.index.get_level_values(0).nunique()} individuals × "
      f"{df.index.get_level_values(1).nunique()} years")
print(df.head())

## 14.1 Fixed Effects Estimation

## Step 3 – Fixed Effects (Within) Estimator

**Within transformation** (time-demean each individual):
$$\ddot{y}_{it} = y_{it} - \bar{y}_i, \quad \ddot{\mathbf{x}}_{it} = \mathbf{x}_{it} - \bar{\mathbf{x}}_i$$

Then OLS on the demeaned data:
$$\ddot{y}_{it} = \ddot{\mathbf{x}}_{it}'\boldsymbol{\beta} + \ddot{u}_{it}$$

The individual effect $a_i$ is **eliminated** by demeaning. FE is consistent even when $a_i$ is correlated with $\mathbf{x}_{it}$, but it **cannot** estimate coefficients of time-invariant regressors.

In [ ]:
fe_model = PanelOLS.from_formula(
    'lwage ~ educ + black + hisp + exper + I(exper**2) + married + union + EntityEffects',
    data=df
).fit()
print(fe_model.summary)

## 14.2 Random Effects

## Step 4 – Random Effects Estimator

If $\text{Cov}(a_i, \mathbf{x}_{it})=0$, RE uses GLS (partial demeaning):

$$\tilde{y}_{it} = y_{it} - \theta\bar{y}_i, \quad \theta = 1 - \sqrt{\frac{\sigma_u^2}{\sigma_u^2 + T\sigma_a^2}}$$

RE is **more efficient** than FE under its assumptions, and it **can** estimate time-invariant variable effects.

**Hausman test** — discriminates between FE and RE:
$$H = (\hat{\boldsymbol{\beta}}_{FE}-\hat{\boldsymbol{\beta}}_{RE})'[\widehat{\text{Var}}(\hat{\boldsymbol{\beta}}_{FE})-\widehat{\text{Var}}(\hat{\boldsymbol{\beta}}_{RE})]^{-1}(\hat{\boldsymbol{\beta}}_{FE}-\hat{\boldsymbol{\beta}}_{RE}) \sim \chi^2_k$$

Rejection of H₀ (p<0.05) → use FE (RE is inconsistent).

In [ ]:
re_model = RandomEffects.from_formula(
    'lwage ~ educ + black + hisp + exper + I(exper**2) + married + union',
    data=df
).fit()
print(re_model.summary)

## Hausman Test (FE vs RE)

In [ ]:
# Compare via linearmodels
comparison = compare({'Pooled OLS': PooledOLS.from_formula('lwage ~ educ + exper + union', data=df).fit(),
                      'Fixed Effects': fe_model,
                      'Random Effects': re_model})
print(comparison)

## 14.4 Clustered Standard Errors

In [ ]:
fe_clustered = PanelOLS.from_formula(
    'lwage ~ exper + I(exper**2) + married + union + EntityEffects',
    data=df
).fit(cov_type='clustered', cluster_entity=True)
print("Fixed Effects with Clustered SEs:")
print(fe_clustered.summary)